
 Notebook du traitement des données pour en faire des data £utilisables
 pour les traitements futures.

In [1]:
# Import the requiered libraries
from binascii import Error
from collections import defaultdict
import os
from pathlib import Path
import json
import sys
from gensim.models.word2vec import Word2Vec
from gensim.parsing.preprocessing import remove_stopwords
from matplotlib.pyplot import ylabel
import missingno as msno
from tensorflow.python.util.nest import _yield_value
from tqdm.notebook import tqdm
import logging  
logging.basicConfig(
                    format= "%(levelname)s - %(asctime)s: %(message)s", 
                    datefmt= '%H:%M:%S', 
                    level= logging.INFO
)
import multiprocessing

import pandas as pd
import numpy as np
import re
import gensim
from gensim.parsing.porter import PorterStemmer
import spacy
nlp = spacy.load("en_core_web_sm")


%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.preprocessing import LabelBinarizer

import keras
from keras.preprocessing.text import Tokenizer
from keras_preprocessing.text import tokenizer_from_json
from keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import Embedding, Dropout 
from keras.layers import Dense, Flatten
from keras.layers import LSTM, Bidirectional 
from keras.layers import Input
from keras.layers import Activation
from keras.models import Model
from keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer
import keras
from sklearn.preprocessing import scale
#from keras.layers.embeddings import Embedding
import pickle


In [2]:
# Define constants
FILE_TWEETS = 'sentiment140\\training.1600000.processed.noemoticon.csv'
FILE_DF_TRAIN = 'df_train_5000'
FILE_DF_VALIDATE = 'df_validate_1000'
FILE_DF_TEST = 'df_test_1000'
FILE_DF_TWEETS = 'df_tweets'


FILE_DICTIONARY = 'dict_simple'
FILE_WORD2VEC_CBOW = "word2vec_CBOW"
FILE_WORD2VEC_SKIPGRAM = "word2vec_Skip_Gram"


# Word2Vec kind
W2V_KIND = {'CBOW' : 0, 'SKIP_GRAM' : 1} 


# Dictionary size
TOP_WORDS = 5_000
# Max length of the embedding vectors. 
EMBEDDING_LENGTH = 100

# Create the simple dictionary even if it already exists
CREATE_SIMPLE_DICTIONARY = True

# Smiley - http://www.emoticonr.com/emoticons/
SMILEYS = {
            r"O:)"  : ':angel:',
            r"O:-)" : ':angel:',
            r"X("   : ':angry:',
            r">:-(" : ':angry:',
            r">:("  : ':angry:',
            r"X("   : ':angry:',
            r":)"   : ':smile:', 
            r":-)"  : ':smile:',
            r":(("  : ':crying:', 
            r":("   : ':frown:', 
            r"=(("  : ':broken heart',
            r"=("   : ':frown:', 
            r":-("  : ':frown:',
            r":["   : ':frown:', 
            r";)"   : ':winking:', 
            r";^)"  : ':winking:', 
            r";-)"  : ':winking:',
            r",)"   : ':winking:'  , 
            r",-)"  : ':winking:',
            r";p"   : ':sticking tongue out:', 
            r";-p"  : ':sticking tongue out:',
            r":p"   : '::sticking tongue out:', 
            r":-p"  : '::sticking tongue out::',
            r":|"   : ':straight face:', 
            r":-|"  : ':straight face:',
            r"O.o"  : ':confuse:', 
            r" :/"  : ':confuse:',  # Need space due to http://
            r":-/"  : ':confuse:',
            r":'("  : ':crying:', 
            r":'-(" : ':crying:',
            r"<3"   : ':love:',
            r"</3"  : ':broken heart:',
            r":D"   : ':grin:',
            r":-D"  : ':grin:',
            r":*"   : ':kiss:',
            r":-*"  : ':kiss:',
            r":-X"  : ':kiss:',
            r":x"   : ':love:',
}

WANTED_WORKERS = os.cpu_count() - 1
 
ADDED_STOPWORDS = set(['i', 'to'])
STOP_WORDS = gensim.parsing.preprocessing.STOPWORDS.union(ADDED_STOPWORDS)
# We want the negative so we keep no and not
REMOVED_STOPWORDS = set(['no', 'not'])
STOP_WORDS = STOP_WORDS.difference(REMOVED_STOPWORDS)

WORD2VEC_WINDOW = 3


In [3]:
# Read the data
path = os.getcwd() + "\\data\\"

df_tweets = pd.read_csv(
                        Path('./data/')/f"{FILE_TWEETS}",
#                        path + FILE_TWEETS,
                        names= [
                                'sentiment_score',
                                'id',
                                'date',
                                'query',
                                'user',
                                'tweet'
                        ],
                        encoding= 'latin',
)

df_tweets.drop(
                columns= ['id', 'date', 'query', 'user'],
                inplace= True
)

df_tweets['pos'] = df_tweets['sentiment_score'].apply(
                                                lambda x: x if x == 0 else 1
)
df_tweets['neg'] = df_tweets['pos'].apply(lambda x: 0 if x == 1 else 1)



 Nous créons les différents type de transformations et les sauvegardons dans
 des features

In [4]:
# Define the functions to work the tweets
def smileys_to_words(tweet):
    for k,v in SMILEYS.items():
        tweet = tweet.replace(k, v)
    return tweet

def clean_tweet(tweet):
    # Remove the @....
    tmp_tweet = re.sub(
                        r'((@)[^ ]*)',
                        '',
                        tweet
    )

    # Delete the http:// and https://
    tmp_tweet = re.sub(
                        r'((http(s|)://)[^ ]*)',
                        '',
                        tmp_tweet
    )

    # Go for lower case
    tmp_tweet = tmp_tweet.lower()

    # delete letters that are repeated more than twice
    tmp_tweet = re.sub(
                        r'(.)\1{2,}', 
                        r'\1', 
                        tmp_tweet
    )

    # changes "n't" to " not"
    tmp_tweet = re.sub(
                        r"(n't)",
                        " not",
                        tmp_tweet
    )

    # remove the "'re" "'s" "'d" as they are to be stop words
    tmp_tweet = re.sub(
                        r"('(re|d|s|m|ll))",
                        "",
                        tmp_tweet
    )

    # keep only the alphabet + # plus :.....: that were smileys
    # and remove the stopwords
    tokens = [
                token
                for token in re.findall(
                                            pattern= r"[a-z]+|#[a-z]+|:[a-z]+:",
                                            string= tmp_tweet,
                                            flags= re.MULTILINE
                )
                if token not in STOP_WORDS
    ]
    return " ".join(tokens)

# Fist pipeline
def pipeline_1(tweet, progress_bar= None):
    if progress_bar is not None:
        progress_bar.update()
    tmp_tweet = smileys_to_words(tweet= tweet)
    tmp_tweet = clean_tweet(tmp_tweet)
    return tmp_tweet

# Stemming
def stem_tweet(tweet, stemmer= None, progress_bar= None):
    if progress_bar is not None:
        progress_bar.update()
    if stemmer is None:
        stemmer = PorterStemmer()
    return stemmer.stem_sentence(tweet)

# Lemmatization
def lemma_tweet(tweet, progress_bar= None):
    if progress_bar is not None:
        progress_bar.update()
    return " ".join([token.lemma_ for token in nlp(tweet)])



# Word to figure : create the dictionary
def word2figure_dict(df, file_name_dict):
    if (os.path.exists(Path('./data/')/f"{file_name_dict}")):
        with open(Path('./data/')/f"{file_name_dict}", 'r') as json_file:
            data = json.load(json_file)
            tokenizer = tokenizer_from_json(data)
    else:
        # We have to create the tokenizer. 
        tokenizer = Tokenizer(
                                lower= True,
                                split= ' '
        )
        tokenizer.fit_on_texts(df.values)
        with open(Path('./data/')/f"{file_name_dict}", 'w') as json_file:
            json_file.write(
                            json.dumps(
                                        tokenizer.to_json(),
                                        ensure_ascii= False
                            )
            )
    return tokenizer

# Word to figure : tweet
def word2figure_tweet(tweet, tokenizer, progress_bar= None):
    if progress_bar is not None:
        progress_bar.update()
    tmp = tokenizer.texts_to_sequences(tweet)
    # Flat and turn unknown into 2
    tmp = [
            array_num[0] + 3
            if len(array_num) == 1 else 2
            for array_num in tmp
    ]
    # Add start sentence 0, and end sentence 1
    result = [0] + tmp + [1]
    return result


# Word2Vec - Create the dict
def word2vec_dict(df, file_name_dict, kind= W2V_KIND['CBOW']):
    if os.path.exists(Path('./data/')/f"{file_name_dict}"):
        w2v_model = Word2Vec.load(os.getcwd() + f"\data\{file_name_dict}")
    else:
        # We have to create the Word2Vec. 
        w2v_model = Word2Vec(
                                sentences= [
                                            tweet.split(' ') 
                                            for tweet in df
                                ],
                                min_count= 1,
                                vector_size= TOP_WORDS,
                                workers= WANTED_WORKERS,
                                window= WORD2VEC_WINDOW,
                                sg= kind                       # CBOW
        )
        w2v_model.save(os.getcwd() + f"\data\{FILE_WORD2VEC_CBOW}.model")
    return w2v_model

# Word2Vec - create the vectors
def tweet_to_w2v(sentence, w2v_model, w2v_model_vocab, progress_bar= None):
    if progress_bar is not None:
        progress_bar.update()
    tmp = []
    if type(w2v_model) == gensim.models.keyedvectors.KeyedVectors:
        tmp = [
                np.mean(w2v_model[token]) 
                if token in w2v_model_vocab else 2
                for token in sentence.split(' ')
        ]
    else:
        tmp = [
                np.mean(w2v_model.wv[token]) 
                if token in w2v_model_vocab else 2
                for token in sentence.split(' ')
        ]
    result = [0] + tmp + [1]
    return result
    

In [5]:
# clean the tweets
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'treatments',
                    leave= False
)
df_tweets['clean'] = [
                        pipeline_1(
                                    tweet= tweet, 
                                    progress_bar= progress_bar
                        )
                        for tweet in df_tweets['tweet']
]
progress_bar.close()


treatments:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [6]:
df_tweets.head()

,sentiment_score,tweet,pos,neg,clean
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...
2,0,@Kenichan I dived many times for the ball. Man...,0,1,dived times ball managed save rest bounds
3,0,my whole body feels itchy and like its on fire,0,1,body feels itchy like
4,0,"@nationwideclass no, it's not behaving at all....",0,1,no not behaving mad ca not


In [7]:
# Word to figure of clean tweets
tokenizer = word2figure_dict(
                                df= df_tweets['clean'],
                                file_name_dict= "Word_2_Figure_Clean.json"
)
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'w2f',
                    leave= False
)
df_tweets['w2f_clean'] = [
                            word2figure_tweet(
                                                tweet= tweet,
                                                tokenizer= tokenizer, 
                                                progress_bar= progress_bar
                            )
                            for tweet in df_tweets['clean']
]
progress_bar.close()



w2f:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [8]:
df_tweets.head()

,sentiment_score,tweet,pos,neg,clean,w2f_clean
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,..."
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...,"[0, 17, 308, 156, 284, 182, 2, 256, 22020, 2, ..."
2,0,@Kenichan I dived many times for the ball. Man...,0,1,dived times ball managed save rest bounds,"[0, 178, 26153, 585, 284, 178, 2, 182, 26153, ..."
3,0,my whole body feels itchy and like its on fire,0,1,body feels itchy like,"[0, 150, 118, 178, 318, 2, 490, 284, 284, 355,..."
4,0,"@nationwideclass no, it's not behaving at all....",0,1,no not behaving mad ca not,"[0, 99, 118, 2, 99, 118, 182, 2, 150, 284, 660..."


In [9]:
# Stemming
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'stem',
                    leave= False
)
stemmer = PorterStemmer()
df_tweets['stem'] = [
                        stem_tweet(
                                    tweet= tweet,
                                    stemmer= stemmer, 
                                    progress_bar= progress_bar
                        )
                        for tweet in df_tweets['clean']
]
progress_bar.close()

tokenizer = word2figure_dict(
                                df= df_tweets['stem'],
                                file_name_dict= "Word_2_Figure_Stem.json"
)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'w2f(stem)',
                    leave= False
)
df_tweets['w2f_stem'] = [
                            word2figure_tweet(
                                                tweet= tweet,
                                                tokenizer= tokenizer, 
                                                progress_bar= progress_bar
                            )
                            for tweet in df_tweets['stem']
]
progress_bar.close()


stem:   0%|          | 0/1600000 [00:00<?, ?it/s]

w2f(stem):   0%|          | 0/1600000 [00:00<?, ?it/s]

In [10]:
df_tweets.head()

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,..."
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...,"[0, 17, 308, 156, 284, 182, 2, 256, 22020, 2, ...",upset ca not updat facebook text result school...,"[0, 20, 327, 173, 308, 202, 2, 277, 14866, 2, ..."
2,0,@Kenichan I dived many times for the ball. Man...,0,1,dived times ball managed save rest bounds,"[0, 178, 26153, 585, 284, 178, 2, 182, 26153, ...",dive time ball manag save rest bound,"[0, 199, 11587, 598, 308, 2, 202, 11587, 165, ..."
3,0,my whole body feels itchy and like its on fire,0,1,body feels itchy like,"[0, 150, 118, 178, 318, 2, 490, 284, 284, 355,...",bodi feel itchi like,"[0, 167, 139, 199, 11587, 2, 498, 308, 308, 37..."
4,0,"@nationwideclass no, it's not behaving at all....",0,1,no not behaving mad ca not,"[0, 99, 118, 2, 99, 118, 182, 2, 150, 284, 660...",no not behav mad ca not,"[0, 120, 139, 2, 120, 139, 202, 2, 167, 308, 6..."


In [11]:
# Lemmatization
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'lemma',
                    leave= False
)
df_tweets['lemma'] = [
                        lemma_tweet(
                                    tweet= tweet,
                                    progress_bar= progress_bar
                        )
                        for tweet in df_tweets['clean']
]
progress_bar.close()

tokenizer = word2figure_dict(
                                df= df_tweets['lemma'],
                                file_name_dict= "Word_2_Figure_Lemma.json"
)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'w2f(lemma)',
                    leave= False
)
df_tweets['w2f_lemma'] = [
                            word2figure_tweet(
                                                tweet= tweet,
                                                tokenizer= tokenizer, 
                                                progress_bar= progress_bar
                            )
                            for tweet in df_tweets['lemma']
]
progress_bar.close()



lemma:   0%|          | 0/1600000 [00:00<?, ?it/s]

w2f(lemma):   0%|          | 0/1600000 [00:00<?, ?it/s]

In [12]:
df_tweets.head()

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2..."
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...,"[0, 17, 308, 156, 284, 182, 2, 256, 22020, 2, ...",upset ca not updat facebook text result school...,"[0, 20, 327, 173, 308, 202, 2, 277, 14866, 2, ...",upset ca not update facebook texting result sc...,"[0, 23, 323, 50, 295, 200, 2, 277, 1029, 2, 12..."
2,0,@Kenichan I dived many times for the ball. Man...,0,1,dived times ball managed save rest bounds,"[0, 178, 26153, 585, 284, 178, 2, 182, 26153, ...",dive time ball manag save rest bound,"[0, 199, 11587, 598, 308, 2, 202, 11587, 165, ...",dive time ball manage save rest bound,"[0, 154, 20, 571, 295, 2, 200, 20, 16, 295, 2,..."
3,0,my whole body feels itchy and like its on fire,0,1,body feels itchy like,"[0, 150, 118, 178, 318, 2, 490, 284, 284, 355,...",bodi feel itchi like,"[0, 167, 139, 199, 11587, 2, 498, 308, 308, 37...",body feel itchy like,"[0, 172, 145, 154, 273, 2, 433, 295, 295, 365,..."
4,0,"@nationwideclass no, it's not behaving at all....",0,1,no not behaving mad ca not,"[0, 99, 118, 2, 99, 118, 182, 2, 150, 284, 660...",no not behav mad ca not,"[0, 120, 139, 2, 120, 139, 202, 2, 167, 308, 6...",no not behave mad ca not,"[0, 125, 145, 2, 125, 145, 200, 2, 172, 295, 6..."


In [13]:
# Word 2 Vec on clean tweet
w2v_model = word2vec_dict(
                            df= df_tweets['clean'],
                            file_name_dict= "w2v_CBOW_model_clean.model",
                            kind= W2V_KIND['CBOW']
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.wv.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V CBOW Clean',
                    leave= False
)
df_tweets['w2v_CBOW_clean'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['clean']
]
progress_bar.close()


ples, 83481 words/s, in_qsize 13, out_qsize 1
INFO - 12:15:42: EPOCH 4 - PROGRESS: at 27.74% examples, 83723 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:43: EPOCH 4 - PROGRESS: at 28.60% examples, 83333 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:44: EPOCH 4 - PROGRESS: at 29.54% examples, 83419 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:46: EPOCH 4 - PROGRESS: at 30.48% examples, 83340 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:47: EPOCH 4 - PROGRESS: at 31.43% examples, 83526 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:48: EPOCH 4 - PROGRESS: at 32.36% examples, 83490 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:49: EPOCH 4 - PROGRESS: at 33.29% examples, 83569 words/s, in_qsize 14, out_qsize 0
INFO - 12:15:50: EPOCH 4 - PROGRESS: at 34.24% examples, 83528 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:51: EPOCH 4 - PROGRESS: at 35.17% examples, 83611 words/s, in_qsize 13, out_qsize 0
INFO - 12:15:52: EPOCH 4 - PROGRESS: at 36.10% examples, 83743 words/s, in_qsize 

W2V CBOW Clean:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [14]:
df_tweets.head()

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928..."
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...,"[0, 17, 308, 156, 284, 182, 2, 256, 22020, 2, ...",upset ca not updat facebook text result school...,"[0, 20, 327, 173, 308, 202, 2, 277, 14866, 2, ...",upset ca not update facebook texting result sc...,"[0, 23, 323, 50, 295, 200, 2, 277, 1029, 2, 12...","[0, -0.002465699, 0.003585951, -0.0016985107, ..."
2,0,@Kenichan I dived many times for the ball. Man...,0,1,dived times ball managed save rest bounds,"[0, 178, 26153, 585, 284, 178, 2, 182, 26153, ...",dive time ball manag save rest bound,"[0, 199, 11587, 598, 308, 2, 202, 11587, 165, ...",dive time ball manage save rest bound,"[0, 154, 20, 571, 295, 2, 200, 20, 16, 295, 2,...","[0, 6.107628e-07, -0.0014740431, -0.0037854344..."
3,0,my whole body feels itchy and like its on fire,0,1,body feels itchy like,"[0, 150, 118, 178, 318, 2, 490, 284, 284, 355,...",bodi feel itchi like,"[0, 167, 139, 199, 11587, 2, 498, 308, 308, 37...",body feel itchy like,"[0, 172, 145, 154, 273, 2, 433, 295, 295, 365,...","[0, -0.0010605041, -0.00599454, -0.0014645172,..."
4,0,"@nationwideclass no, it's not behaving at all....",0,1,no not behaving mad ca not,"[0, 99, 118, 2, 99, 118, 182, 2, 150, 284, 660...",no not behav mad ca not,"[0, 120, 139, 2, 120, 139, 202, 2, 167, 308, 6...",no not behave mad ca not,"[0, 125, 145, 2, 125, 145, 200, 2, 172, 295, 6...","[0, -0.0020759227, -0.0016985107, -0.000430661..."


In [15]:
# Word 2 Vec for stem
w2v_model = word2vec_dict(
                            df= df_tweets['stem'],
                            file_name_dict= "w2v_CBOW_model_stem.model",
                            kind= W2V_KIND['CBOW']
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.wv.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V CBOW Stem',
                    leave= False
)
df_tweets['w2v_CBOW_stem'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['stem']
]
progress_bar.close()


mples, 83048 words/s, in_qsize 13, out_qsize 0
INFO - 12:31:52: EPOCH 4 - PROGRESS: at 28.03% examples, 82690 words/s, in_qsize 14, out_qsize 0
INFO - 12:31:53: EPOCH 4 - PROGRESS: at 28.98% examples, 82876 words/s, in_qsize 13, out_qsize 0
INFO - 12:31:54: EPOCH 4 - PROGRESS: at 29.91% examples, 83024 words/s, in_qsize 14, out_qsize 0
INFO - 12:31:55: EPOCH 4 - PROGRESS: at 30.67% examples, 82661 words/s, in_qsize 13, out_qsize 0
INFO - 12:31:56: EPOCH 4 - PROGRESS: at 31.80% examples, 83110 words/s, in_qsize 14, out_qsize 0
INFO - 12:31:57: EPOCH 4 - PROGRESS: at 32.54% examples, 82739 words/s, in_qsize 13, out_qsize 0
INFO - 12:31:58: EPOCH 4 - PROGRESS: at 33.67% examples, 83139 words/s, in_qsize 13, out_qsize 0
INFO - 12:31:59: EPOCH 4 - PROGRESS: at 34.52% examples, 82773 words/s, in_qsize 14, out_qsize 0
INFO - 12:32:01: EPOCH 4 - PROGRESS: at 35.54% examples, 82936 words/s, in_qsize 13, out_qsize 0
INFO - 12:32:02: EPOCH 4 - PROGRESS: at 36.38% examples, 82717 words/s, in_qsize

W2V CBOW Stem:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [16]:
df_tweets.head(2)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-..."
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...,"[0, 17, 308, 156, 284, 182, 2, 256, 22020, 2, ...",upset ca not updat facebook text result school...,"[0, 20, 327, 173, 308, 202, 2, 277, 14866, 2, ...",upset ca not update facebook texting result sc...,"[0, 23, 323, 50, 295, 200, 2, 277, 1029, 2, 12...","[0, -0.002465699, 0.003585951, -0.0016985107, ...","[0, 0.0017395355, 0.00041537866, -0.0008892282..."


In [17]:
# Word 2 Vec for lemma
w2v_model = word2vec_dict(
                            df= df_tweets['lemma'],
                            file_name_dict= "w2v_CBOW_model_lemma.model",
                            kind= W2V_KIND['CBOW']
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.wv.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V CBOW Lemma',
                    leave= False
)
df_tweets['w2v_CBOW_lemma'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['lemma']
]
progress_bar.close()


ples, 85191 words/s, in_qsize 12, out_qsize 1
INFO - 12:48:30: EPOCH 4 - PROGRESS: at 27.55% examples, 85765 words/s, in_qsize 14, out_qsize 0
INFO - 12:48:31: EPOCH 4 - PROGRESS: at 28.39% examples, 85168 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:33: EPOCH 4 - PROGRESS: at 29.59% examples, 85669 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:34: EPOCH 4 - PROGRESS: at 30.51% examples, 85737 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:35: EPOCH 4 - PROGRESS: at 31.53% examples, 85692 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:36: EPOCH 4 - PROGRESS: at 32.52% examples, 85928 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:37: EPOCH 4 - PROGRESS: at 33.44% examples, 85614 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:38: EPOCH 4 - PROGRESS: at 34.54% examples, 85935 words/s, in_qsize 13, out_qsize 0
INFO - 12:48:39: EPOCH 4 - PROGRESS: at 35.35% examples, 85655 words/s, in_qsize 14, out_qsize 0
INFO - 12:48:40: EPOCH 4 - PROGRESS: at 36.45% examples, 85974 words/s, in_qsize 

W2V CBOW Lemma:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [18]:
df_tweets.head(2)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851..."
1,0,is upset that he can't update his Facebook by ...,0,1,upset ca not update facebook texting result sc...,"[0, 17, 308, 156, 284, 182, 2, 256, 22020, 2, ...",upset ca not updat facebook text result school...,"[0, 20, 327, 173, 308, 202, 2, 277, 14866, 2, ...",upset ca not update facebook texting result sc...,"[0, 23, 323, 50, 295, 200, 2, 277, 1029, 2, 12...","[0, -0.002465699, 0.003585951, -0.0016985107, ...","[0, 0.0017395355, 0.00041537866, -0.0008892282...","[0, 9.745369e-05, -0.00423395, 0.0009797486, -..."


In [19]:
# Word 2 Vec Skip Gram on clean tweet
w2v_model = word2vec_dict(
                            df= df_tweets['clean'],
                            file_name_dict= "w2v_SG_model_clean.model",
                            kind= W2V_KIND['SKIP_GRAM']
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.wv.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V SG Clean',
                    leave= False
)
df_tweets['w2v_SG_clean'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['clean']
]
progress_bar.close()


ze 0
INFO - 13:18:07: EPOCH 5 - PROGRESS: at 14.66% examples, 32270 words/s, in_qsize 13, out_qsize 0
INFO - 13:18:09: EPOCH 5 - PROGRESS: at 15.22% examples, 32251 words/s, in_qsize 13, out_qsize 0
INFO - 13:18:10: EPOCH 5 - PROGRESS: at 15.50% examples, 32121 words/s, in_qsize 14, out_qsize 0
INFO - 13:18:11: EPOCH 5 - PROGRESS: at 15.98% examples, 32259 words/s, in_qsize 14, out_qsize 0
INFO - 13:18:13: EPOCH 5 - PROGRESS: at 16.55% examples, 32357 words/s, in_qsize 14, out_qsize 0
INFO - 13:18:14: EPOCH 5 - PROGRESS: at 16.91% examples, 32372 words/s, in_qsize 13, out_qsize 0
INFO - 13:18:15: EPOCH 5 - PROGRESS: at 17.29% examples, 32298 words/s, in_qsize 14, out_qsize 0
INFO - 13:18:16: EPOCH 5 - PROGRESS: at 17.75% examples, 32506 words/s, in_qsize 13, out_qsize 0
INFO - 13:18:18: EPOCH 5 - PROGRESS: at 17.94% examples, 32172 words/s, in_qsize 13, out_qsize 0
INFO - 13:18:19: EPOCH 5 - PROGRESS: at 18.41% examples, 32404 words/s, in_qsize 13, out_qsize 1
INFO - 13:18:20: EPOCH 5 

W2V SG Clean:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [20]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334..."


In [21]:
# Word 2 Vec for stem
w2v_model = word2vec_dict(
                            df= df_tweets['stem'],
                            file_name_dict= "w2v_SG_model_stem.model",
                            kind= W2V_KIND['CBOW']
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.wv.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V SG Stem',
                    leave= False
)
df_tweets['w2v_SG_stem'] = [
                            tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                            )
                            for sentence in df_tweets['stem']
]
progress_bar.close()


t 25.55% examples, 87518 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:01: EPOCH 4 - PROGRESS: at 26.69% examples, 87501 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:02: EPOCH 4 - PROGRESS: at 27.83% examples, 87946 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:03: EPOCH 4 - PROGRESS: at 28.79% examples, 87947 words/s, in_qsize 14, out_qsize 0
INFO - 13:38:04: EPOCH 4 - PROGRESS: at 29.82% examples, 88197 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:05: EPOCH 4 - PROGRESS: at 30.86% examples, 87901 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:06: EPOCH 4 - PROGRESS: at 31.99% examples, 88453 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:07: EPOCH 4 - PROGRESS: at 32.82% examples, 87822 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:09: EPOCH 4 - PROGRESS: at 34.15% examples, 88049 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:10: EPOCH 4 - PROGRESS: at 35.44% examples, 88341 words/s, in_qsize 13, out_qsize 0
INFO - 13:38:11: EPOCH 4 - PROGRESS: at 36.47% examples, 88180 words

W2V SG Stem:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [22]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167..."


In [23]:
# Word 2 Vec for lemma
w2v_model = word2vec_dict(
                            df= df_tweets['lemma'],
                            file_name_dict= "w2v_SG_model_lemma.model",
                            kind= W2V_KIND['SKIP_GRAM']
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.wv.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V SG Lemma',
                    leave= False
)
df_tweets['w2v_SG_lemma'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['lemma']
]
progress_bar.close()


ze 0
INFO - 16:43:10: EPOCH 5 - PROGRESS: at 18.35% examples, 31135 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:11: EPOCH 5 - PROGRESS: at 18.91% examples, 31209 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:13: EPOCH 5 - PROGRESS: at 19.37% examples, 31219 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:14: EPOCH 5 - PROGRESS: at 19.91% examples, 31358 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:15: EPOCH 5 - PROGRESS: at 20.36% examples, 31555 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:16: EPOCH 5 - PROGRESS: at 20.64% examples, 31350 words/s, in_qsize 14, out_qsize 0
INFO - 16:43:18: EPOCH 5 - PROGRESS: at 21.10% examples, 31379 words/s, in_qsize 14, out_qsize 0
INFO - 16:43:19: EPOCH 5 - PROGRESS: at 21.64% examples, 31377 words/s, in_qsize 14, out_qsize 0
INFO - 16:43:21: EPOCH 5 - PROGRESS: at 22.10% examples, 31422 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:22: EPOCH 5 - PROGRESS: at 22.65% examples, 31631 words/s, in_qsize 13, out_qsize 0
INFO - 16:43:23: EPOCH 5 

W2V SG Lemma:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [24]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994..."


In [25]:
# Word 2 Vec with Google
wordmodelfile=Path('./data/')/"GoogleNews-vectors-negative300.bin.gz"
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(
                                                            wordmodelfile,
                                                            binary=True
)
# Create a set so the work is faster
w2v_model_vocab = set(w2v_model.index_to_key)

progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V Google Clean',
                    leave= False
)
df_tweets['w2v_Google_clean'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['clean']
]
progress_bar.close()


INFO - 16:52:57: loading projection weights from data\GoogleNews-vectors-negative300.bin.gz
INFO - 16:54:13: KeyedVectors lifecycle event {'msg': 'loaded (3000000, 300) matrix of type float32 from data\\GoogleNews-vectors-negative300.bin.gz', 'binary': True, 'encoding': 'utf8', 'datetime': '2021-06-16T16:54:13.204176', 'gensim': '4.0.1', 'python': '3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19042-SP0', 'event': 'load_word2vec_format'}


W2V Google Clean:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [26]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma,w2v_Google_clean
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994...","[0, 0.003761495, -0.0021979213, 0.0020484924, ..."


In [27]:
# Word 2 Vec for stem
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V Google Stem',
                    leave= False
)
df_tweets['w2v_Google_stem'] = [
                                tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                )
                                for sentence in df_tweets['stem']
]
progress_bar.close()


W2V Google Stem:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [28]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma,w2v_Google_clean,w2v_Google_stem
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ..."


In [29]:
# Word 2 Vec for lemma
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= 'W2V Google Lemma',
                    leave= False
)
df_tweets['w2v_Google_lemma'] = [
                                    tweet_to_w2v(
                                            sentence= sentence,
                                            w2v_model= w2v_model,
                                            w2v_model_vocab= w2v_model_vocab, 
                                            progress_bar= progress_bar
                                    )
                                    for sentence in df_tweets['lemma']
]
progress_bar.close()


W2V Google Lemma:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [30]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma,w2v_Google_clean,w2v_Google_stem,w2v_Google_lemma
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ..."


In [138]:
def build_word_vector(sentence, w2v_model, tfidf, progress_bar= None):
    if progress_bar is not None:
        progress_bar.update()
    tmp = []
    if type(w2v_model) == gensim.models.keyedvectors.KeyedVectors:
        tmp = [
                np.mean(w2v_model[token] * tfidf[token]) 
                if (token in w2v_model_vocab) and (token in tfidf.keys()) else 2
                for token in sentence.split(' ')
        ]
    else:
        tmp = [
                np.mean(w2v_model.wv[token] * tfidf[token])
                if (token in w2v_model_vocab) and (token in tfidf.keys()) else 2
                for token in sentence.split(' ')
        ]
    result = [0] + tmp + [1]
    return result

In [147]:
# Word2Vec with TF.IDF as weight Clean
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(df_tweets['clean'].values)
tfidf = dict(
                zip(vectorizer.get_feature_names(), vectorizer.idf_)
)
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= "clean W2v TFIDF",
                    leave= False
)
tmp = [
        build_word_vector(
                            sentence= sentence,
                            w2v_model= w2v_model,
                            tfidf= tfidf,
                            progress_bar= progress_bar
        )
        for sentence in df_tweets['clean'].values
]
df_tweets['w2v_tf.idf_clean'] = tmp
progress_bar.close()

clean W2v TFIDF:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [148]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,w2v_CBOW_clean,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma,w2v_Google_clean,w2v_Google_stem,w2v_Google_lemma,w2v_tf.idf_clean
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...","[0, 0.00019046631, -0.00066044676, -0.00085928...","[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.021935537, -0.017436262, 0.019266034, 0...."


In [149]:
# Word2Vec with TF.IDF as weight Stem
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(df_tweets['stem'].values)
tfidf = dict(
                zip(vectorizer.get_feature_names(), vectorizer.idf_)
)
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= "stem w2v TFIDF",
                    leave= False
)
df_tweets['w2v_tf.idf_stem'] = [
                                build_word_vector(
                                                    sentence= sentence,
                                                    w2v_model= w2v_model,
                                                    tfidf= tfidf,
                                                    progress_bar= progress_bar
                                )
                                for sentence in df_tweets['stem']
]
progress_bar.close()

stem w2v TFIDF:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [150]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,...,w2v_CBOW_stem,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma,w2v_Google_clean,w2v_Google_stem,w2v_Google_lemma,w2v_tf.idf_clean,w2v_tf.idf_stem
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...",...,"[0, -0.0019677565, -0.0021536304, -5.2689553e-...","[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.021935537, -0.017436262, 0.019266034, 0....","[0, 0.021202028, -0.017416617, 0.019266034, 0...."


In [151]:
# Word2Vec with TF.IDF as weight Stem
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform(df_tweets['stem'].values)
tfidf = dict(
                zip(vectorizer.get_feature_names(), vectorizer.idf_)
)
progress_bar = tqdm(
                    total= len(df_tweets),
                    desc= "lemma w2v TFIDF",
                    leave= False
)
df_tweets['w2v_tf.idf_lemma'] = [
                                    build_word_vector(
                                                        sentence= sentence,
                                                        w2v_model= w2v_model,
                                                        tfidf= tfidf,
                                                        progress_bar= progress_bar
                                    )
                                    for sentence in df_tweets['lemma']
]
progress_bar.close()

lemma w2v TFIDF:   0%|          | 0/1600000 [00:00<?, ?it/s]

In [152]:
df_tweets.head(1)

,sentiment_score,tweet,pos,neg,clean,w2f_clean,stem,w2f_stem,lemma,w2f_lemma,...,w2v_CBOW_lemma,w2v_SG_clean,w2v_SG_stem,w2v_SG_lemma,w2v_Google_clean,w2v_Google_stem,w2v_Google_lemma,w2v_tf.idf_clean,w2v_tf.idf_stem,w2v_tf.idf_lemma
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,1,aw bummer shoulda got david carr day d,"[0, 22020, 83, 2, 150, 17, 146, 146, 284, 176,...",aw bummer shoulda got david carr dai d,"[0, 14866, 98, 2, 167, 20, 165, 165, 308, 195,...",aw bummer shoulda get david carr day d,"[0, 1029, 104, 2, 172, 23, 16, 16, 295, 195, 2...",...,"[0, 9.1063026e-05, -0.0008590824, 0.0009834851...","[0, 8.700209e-05, 0.00011658065, 0.00019656334...","[0, -0.001969145, -0.0022075493, -0.0001310167...","[0, 3.3082677e-05, -0.00014922161, -0.00027994...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.003761495, -0.0021979213, 0.0020484924, ...","[0, 0.021935537, -0.017436262, 0.019266034, 0....","[0, 0.021202028, -0.017416617, 0.019266034, 0....","[0, 0.021202028, -0.017416617, 0.019266034, 0...."


In [153]:
df_tweets.to_csv(os.getcwd() + f"\\data\\{FILE_DF_TWEETS}.csv")

In [154]:
# Create the Train, Validate and Test to do the comparaisons
df_train = pd.read_pickle(os.getcwd() + f"\\data\\{FILE_DF_TRAIN}.pkl")
index_train = df_train.index
df_train_augmented = df_tweets.loc[index_train]
df_train_augmented.to_csv(os.getcwd() + f"\\data\\{FILE_DF_TRAIN}-Augmented.csv")
logging.info("CSV saved")

print("Train sauvegardé, passe à Validate")
df_validate = pd.read_pickle(os.getcwd() + f"\\data\\{FILE_DF_VALIDATE}.pkl")
index_validate = df_validate.index
df_validate_augmented = df_tweets.loc[index_validate]
df_train_augmented.to_csv(os.getcwd() + f"\\data\\{FILE_DF_VALIDATE}-Augmented.csv")
logging.info("CSV saved")

print("Validate sauvegardé, passe à Test")
df_test = pd.read_pickle(os.getcwd() + f"\\data\\{FILE_DF_TEST}.pkl")
index_test = df_test.index
df_test_augmented = df_tweets.loc[index_test]
df_validate_augmented = df_tweets.loc[index_validate]
df_train_augmented.to_csv(os.getcwd() + f"\\data\\{FILE_DF_VALIDATE}-Augmented.csv")
logging.info("CSV saved")

print("Passe au Pickles")
df_train_augmented.to_pickle(os.getcwd() + f"\\data\\{FILE_DF_TRAIN}-Augmented.pkl")
logging.info("PKL saved")
df_validate_augmented.to_pickle(os.getcwd() + f"\\data\\{FILE_DF_VALIDATE}-Augmented.pkl")
logging.info("PKL saved")
df_test_augmented.to_pickle(os.getcwd() + f"\\data\\{FILE_DF_TEST}-Augmented.pkl")
logging.info("PKL saved")


INFO - 00:31:52: CSV saved
Train sauvegardé, passe à Validate
INFO - 00:31:53: CSV saved
Validate sauvegardé, passe à Test
INFO - 00:31:53: CSV saved
Passe au Pickles
INFO - 00:31:56: PKL saved
INFO - 00:31:59: PKL saved
INFO - 00:32:01: PKL saved


In [ ]:
# Save everything
with open(os.getcwd() + f"\\data\\{FILE_DF_TWEETS}.pkl", 'wb') as file_pickle:
    pickle.dump(
                df_tweets,
                file_pickle,
                protocol= pickle.HIGHEST_PROTOCOL
    )
